# User-friendly toy generation

This notebook focuses only on pseudo-data generation. Signal-only, efficiency/veto, backgrounds and direct-CP toys all use the same short high-level interface.

There are exactly two public sampling methods: **inverse-transform** and **accept-reject**. Inverse-transform is the default. The inverse-transform path prepares numerical inverse CDFs on the physical Dalitz plane and is especially useful for large samples or repeated toys at fixed truth parameters. Accept-reject remains available explicitly as an independent validation method.


In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    CPToyBackground,CPRealImag,DecayChannel,DecayModel,NonResonant,Parameter,
    RealImag,Resonance,ToyBackground,enable_x64,generate_cp_toy,generate_toy,
    prepare_inverse_toy_generator,plot_dalitz,plot_square_dalitz,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency
enable_x64()


In [ ]:
model=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),
    [
        Resonance("Kstar",(0,2),RealImag(1,0),mass=0.8958,width=0.0474,spin=1),
        NonResonant(RealImag(-0.25,0.10)),
    ],
    normalization_method="square-dalitz",normalization_resolution=160,normalization_pair=(0,2),
)
eff=FunctionalEfficiency(lambda d:0.55+0.30*jnp.clip(d["s13"]/20,0,1))
bkg=FunctionalBackground(lambda d:0.35+0.65*jnp.clip(d["s23"]/25,0,1))

# inverse-transform is the default; no method argument is needed.
signal=generate_toy(model,15_000,seed=1801)
selected=generate_toy(model,15_000,efficiency=eff,seed=1802)
mixture=generate_toy(
    model,20_000,efficiency=eff,signal_fraction=0.80,
    backgrounds=(ToyBackground("comb",bkg),),
    seed=1803,
)


In [ ]:
fig,axes=plt.subplots(1,3,figsize=(16,4.5),constrained_layout=True)
plot_dalitz(signal,x="s13",y="s23",ax=axes[0],title="signal")
plot_dalitz(selected,x="s13",y="s23",ax=axes[1],title="signal + efficiency")
plot_dalitz(mixture,x="s13",y="s23",ax=axes[2],title="signal + efficiency + background")
plt.show()


## Explicit inverse-transform generation

The default can also be selected explicitly. `inverse_resolution` controls the numerical CDF grid:


In [ ]:
inverse_toy = generate_toy(
    model,
    50_000,
    efficiency=eff,
    method="inverse-transform",
    inverse_resolution=512,
    seed=1810,
)
plot_dalitz(inverse_toy,x="s13",y="s23",bins=80,title="inverse-transform toy")
plt.show()


For repeated pseudoexperiments with the same model parameters, prepare the inverse CDFs once. Preparation evaluates the amplitude/efficiency/veto on the grid; each later `generate` call only performs CDF inversion, momentum reconstruction and shuffling.


In [ ]:
prepared = prepare_inverse_toy_generator(
    model,
    efficiency=eff,
    resolution=512,
)
toy_a = prepared.generate(20_000, seed=1820)
toy_b = prepared.generate(20_000, seed=1821)
print(toy_a.size, toy_b.size)


When explicitly using accept-reject, `pool_size` controls the pilot sample used to estimate the envelope. If a generated candidate exceeds that envelope, the component generation restarts with a larger envelope rather than clipping the acceptance probability. `pool_size` and `batch_size` do not apply to inverse-transform generation.


In [ ]:
cp=CPRealImag(0.8,0.2,0.08,-0.05)
plus=DecayModel(
    DecayChannel("B+",("K+","pi+","pi-")),[NonResonant(cp.for_charge(+1))],
    normalization_method="square-dalitz",normalization_resolution=140,normalization_pair=(0,2),
)
minus=DecayModel(
    DecayChannel("B-",("K-","pi-","pi+")),[NonResonant(cp.for_charge(-1))],
    normalization_method="square-dalitz",normalization_resolution=140,normalization_pair=(0,2),
)
plus_toy,minus_toy=generate_cp_toy(
    plus,minus,20_000,plus_efficiency=eff,minus_efficiency=eff,
    signal_fraction=0.85,backgrounds=(CPToyBackground("comb",bkg),),
    method="inverse-transform",inverse_resolution=256,
    seed=1804,
)
print("B+ / B-:",plus_toy.size,minus_toy.size)
